In [1]:
import os
import sys

path = os.path.abspath("")
splitted = path.split(os.sep)
user_independent = os.path.join(
    splitted[0] + os.sep, splitted[1], splitted[2], splitted[3]
)
src_path = os.path.join(user_independent, "GitHub", "Photoswitching")
sys.path.append(src_path)

import src.blinking as bl
import src.distributions as dist
import src.emissions as em
import src.fcs as fcs_p
import src.figure as fi
import src.fluorophores as fl
import src.formulas as fo
import src.miscellaneous as mi
import src.network as net
import src.simulation as si
import src.prediction as pr
import src.analysis as an
import src.transitions as tr

import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2

import warnings


def custom_warning_format(msg, category, filename, lineno, line=None):
    if line is None:
        import linecache

        line = linecache.getline(filename, lineno)
    return f"WARNING for line: {line} {msg} \n"


warnings.formatwarning = custom_warning_format

In [14]:
fluorophores = fl.construct_fluorophores(
    name="cy5_gidi", distance=10, count=3, shape="square"
)

fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)
transitions = fluorophore_system.load_transitions(
    irradiance=2.5,
    wavelength=640,
    bleaching=True,
    energy_transfer=True,
    dstorm=True,
    reducing_agent="mea",
    concentration=100,
    ph=8,
)

cis_fret_1 = tr.Transition(transition_type=tr.TransitionType.CIS_FRET_1,
                           rate=1e9, fluorophore_ids=[(0, 1), (1, 0),
                                                      (0, 2), (2, 0)])
s_s_annihilation = tr.Transition(transition_type=tr.TransitionType.S_S_ANNIHILATION,
                                 rate=1e9, fluorophore_ids=[(0, 1), (1, 0),
                                                            (0, 2), (2, 0)])
transitions['D: cy5_gidi, A: cy5_gidi, dist: 10.0'].extend([cis_fret_1, s_s_annihilation])
cis_fret_1 = tr.Transition(transition_type=tr.TransitionType.CIS_FRET_1,
                           rate=1e8, fluorophore_ids=[(1, 2), (2, 1)])
s_s_annihilation = tr.Transition(transition_type=tr.TransitionType.S_S_ANNIHILATION,
                                 rate=1e8, fluorophore_ids=[(1, 2), (2, 1)])
transitions['D: cy5_gidi, A: cy5_gidi, dist: 14.142'].extend([cis_fret_1, s_s_annihilation])

transition_set = tr.TransitionSet(transitions, fluorophore_system)
transition_set = transition_set.filter_by_identity([15, 21])
transition_set.finalize()

In [15]:
transition_set.transition_df

transition_type  \
Fluorophore                            identity                                           
cy5_gidi                               0                      TransitionType.EXCITATION   
                                       1            TransitionType.FLUORESCENT_EMISSION   
                                       2         TransitionType.INTERSYSTEM_CROSSING_ST   
                                       3         TransitionType.INTERSYSTEM_CROSSING_TS   
                                       4                   TransitionType.ISOMERIZATION   
                                       5               TransitionType.BACKISOMERIZATION   
                                       6           TransitionType.INTERNAL_CONVERSION_S   
                                       7                      TransitionType.ET_CYCLE_T   
                                       8                      TransitionType.ET_CYCLE_S   
                                       9                     TransitionType.REDUCTION_T   
                                       10                    TransitionType.REDUCTION_S   
                                       11                    TransitionType.OXIDATION_1   
                                       12               TransitionType.PHOTOBLEACHING_1   
D: cy5_gidi, A: cy5_gidi, dist: 10.0   13                     TransitionType.CIS_FRET_2   
                                       14                     TransitionType.OFF_FRET_1   
                                       15               TransitionType.S_T_ANNIHILATION   
                                       16                     TransitionType.CIS_FRET_1   
                                       17               TransitionType.S_S_ANNIHILATION   
D: cy5_gidi, A: cy5_gidi, dist: 14.142 18                     TransitionType.CIS_FRET_2   
                                       19                     TransitionType.OFF_FRET_1   
                                       20               TransitionType.S_T_ANNIHILATION   
                                       21                     TransitionType.CIS_FRET_1   
                                       22               TransitionType.S_S_ANNIHILATION   

                                                abbreviation  \
Fluorophore                            identity                
cy5_gidi                               0                 EXC   
                                       1                 FLU   
                                       2               ISCST   
                                       3               ISCTS   
                                       4                 ISO   
                                       5                BISO   
                                       6                 ICS   
                                       7                 ETT   
                                       8                 ETS   
                                       9                REDT   
                                       10               REDS   
                                       11               OXI1   
                                       12               BLE1   
D: cy5_gidi, A: cy5_gidi, dist: 10.0   13             CFRET2   
                                       14             OFRET1   
                                       15                STA   
                                       16             CFRET1   
                                       17                SSA   
D: cy5_gidi, A: cy5_gidi, dist: 14.142 18             CFRET2   
                                       19             OFRET1   
                                       20                STA   
                                       21             CFRET1   
                                       22                SSA   

                                                       initial_state  \
Fluorophore                            identity                        
cy5_gidi                               0           

In [32]:
photon_collection_rate = fo.calculate_photon_collection_rate(NA=1.45, n1=1.51)
photon_collection_rate

0.36045491718075845

In [27]:
simulation = si.Simulation(transition_set=transition_set)
simulation.run(size=1e7)
emis = em.Emissions(frame_time='1ms', bandpass=[655, 731], seed=100)
emis.extract(simulation)
photon_collection_rate = fo.calculate_photon_collection_rate(NA=1.45, n1=1.51)
emis.add_photon_collection_objective(p=photon_collection_rate, seed=100) 
emis.add_quantum_efficiency(p=0.9, seed=100)

In [28]:
emis.event_time_series

0.000    1189
0.001    1231
0.002    1271
0.003    1214
0.004    1219
         ... 
0.243    1249
0.244    1206
0.245    1154
0.246    1195
0.247    1031
Length: 248, dtype: int64